Distribution of NHDAs across CORINE Land Cover Level 3 classes (2012).
Colors from RGB column in CLC_legend.csv.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

PATH_GPKG   = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Comparison_NHDA_RA\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure.gpkg"
PATH_LEGEND = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\CORINE_Land_Cover_DE\CLC_legend.csv"
OUTPUT_DIR  = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Figures\NHDA_CLC")
OUTPUT_PLOT = OUTPUT_DIR / "nhda_clc_level3.jpg"

def rgb_to_hex(rgb_text):
    parts = str(rgb_text).strip().replace(",", "-").split("-")
    parts = [p.strip() for p in parts if p.strip()]
    if len(parts) != 3:
        return "#bdbdbd"
    try:
        r, g, b = [max(0, min(255, int(p))) for p in parts]
        return f"#{r:02x}{g:02x}{b:02x}"
    except Exception:
        return "#bdbdbd"

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
gdf = gpd.read_file(PATH_GPKG)
nhda = gdf[gdf["type"] == "NHDA"].copy()

legend_df = pd.read_csv(PATH_LEGEND, sep=";", dtype={"CLC_CODE": str})
legend_df["CLC_CODE"]     = legend_df["CLC_CODE"].astype(str).str.strip()
legend_df["LABEL3"]       = legend_df["LABEL3"].astype(str).str.strip()
legend_df["clc_code_int"] = legend_df["CLC_CODE"].astype(int)
legend_df["COLOR_HEX"]    = legend_df["RGB"].apply(rgb_to_hex)

nhda["clc_code"] = nhda["nhda_clc_2012"].astype(str).str.strip()

counts = nhda["clc_code"].value_counts().rename_axis("CLC_CODE").reset_index(name="count")
counts = counts.merge(
    legend_df[["CLC_CODE", "clc_code_int", "COLOR_HEX", "LABEL3"]],
    on="CLC_CODE", how="left",
)
counts["COLOR_HEX"]    = counts["COLOR_HEX"].fillna("#bdbdbd")
counts["clc_code_int"] = counts["clc_code_int"].fillna(999).astype(int)
counts["LABEL3"]       = counts["LABEL3"].fillna("Unknown")
counts["y_label"]      = counts["CLC_CODE"] + " - " + counts["LABEL3"]
counts["pct"]          = counts["count"] / counts["count"].sum() * 100
# Sort by CLC code; reverse for barh (lowest code at top)
counts = counts.sort_values("clc_code_int", ascending=False).reset_index(drop=True)

# ---------------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 16,
    "axes.labelsize": 15,
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
})

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(
    counts["y_label"],
    counts["count"],
    color=counts["COLOR_HEX"],
    edgecolor="#333333",
    linewidth=0.6,
    zorder=3,
)

ax.tick_params(axis="y", labelsize=13)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.set_xlabel("Number of NHDAs", fontsize=12)
# ax.set_title("Distribution of NHDAs by CORINE Land Cover Level 3 (2012)", fontsize=12, pad=12)

ax.grid(axis="x", color="#cccccc", linewidth=0.6, zorder=0)
ax.set_axisbelow(True)

for side in ["top", "right"]:
    ax.spines[side].set_visible(False)

x_offset = counts["count"].max() * 0.012
for bar, val in zip(bars, counts["count"]):
    ax.text(
        bar.get_width() + x_offset,
        bar.get_y() + bar.get_height() / 2,
        str(val),
        ha="left", va="center", fontsize=11, color="#333333",
    )

fig.tight_layout()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(OUTPUT_PLOT, bbox_inches="tight", facecolor="white", format="jpg")
plt.show()

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
total = counts["count"].sum()
print(f"Total NHDAs: {total}\n")

# Count + share per CLC class
summary = counts.sort_values("clc_code_int")[["y_label", "count", "pct"]].copy()
summary["pct"] = summary["pct"].map(lambda x: f"{x:.1f}%")
print(summary.to_string(index=False))

# nhda_clc_2012_pct statistics per CLC class
pct_stats = (
    nhda.groupby("clc_code")["nhda_clc_2012_pct"]
    .agg(
        mean="mean",
        std="std",
        min="min",
        p10=lambda x: x.quantile(0.1),
        p90=lambda x: x.quantile(0.9),
        max="max",
    )
    .reset_index()
    .rename(columns={"clc_code": "CLC_CODE"})
    .merge(legend_df[["CLC_CODE", "clc_code_int", "LABEL3"]], on="CLC_CODE", how="left")
    .sort_values("clc_code_int")
)
pct_stats["y_label"] = pct_stats["CLC_CODE"] + " - " + pct_stats["LABEL3"]
pct_stats = pct_stats[["y_label", "mean", "std", "min", "p5", "p95", "max"]]
for col in ["mean", "std", "min", "p10", "p90", "max"]:
    pct_stats[col] = pct_stats[col].map(lambda x: f"{x:.1f}%")

print("\nnhda_clc_2012_pct per CLC class:")
print(pct_stats.to_string(index=False))
print(f"\nSaved: {OUTPUT_PLOT}")


Distribution of NHDA Surrounding Land Cover NHDAs across CORINE Land Cover Level 3 classes (2021).
Colors from RGB column in CLC_legend.csv.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

PATH_GPKG   = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Comparison_NHDA_RA\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure.gpkg"
PATH_LEGEND = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\CORINE_Land_Cover_DE\CLC_legend.csv"
OUTPUT_DIR  = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Figures\NHDA_CLC")
OUTPUT_PLOT = OUTPUT_DIR / "sur_nhda_clc_level3.jpg"

def rgb_to_hex(rgb_text):
    parts = str(rgb_text).strip().replace(",", "-").split("-")
    parts = [p.strip() for p in parts if p.strip()]
    if len(parts) != 3:
        return "#bdbdbd"
    try:
        r, g, b = [max(0, min(255, int(p))) for p in parts]
        return f"#{r:02x}{g:02x}{b:02x}"
    except Exception:
        return "#bdbdbd"

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
gdf = gpd.read_file(PATH_GPKG)
nhda = gdf[gdf["type"] == "NHDA"].copy()

legend_df = pd.read_csv(PATH_LEGEND, sep=";", dtype={"CLC_CODE": str})
legend_df["CLC_CODE"]     = legend_df["CLC_CODE"].astype(str).str.strip()
legend_df["LABEL3"]       = legend_df["LABEL3"].astype(str).str.strip()
legend_df["clc_code_int"] = legend_df["CLC_CODE"].astype(int)
legend_df["COLOR_HEX"]    = legend_df["RGB"].apply(rgb_to_hex)

nhda["clc_code"] = nhda["nhda_sur_clc_2021"].astype(str).str.strip()

counts = nhda["clc_code"].value_counts().rename_axis("CLC_CODE").reset_index(name="count")
counts = counts.merge(
    legend_df[["CLC_CODE", "clc_code_int", "COLOR_HEX", "LABEL3"]],
    on="CLC_CODE", how="left",
)
counts["COLOR_HEX"]    = counts["COLOR_HEX"].fillna("#bdbdbd")
counts["clc_code_int"] = counts["clc_code_int"].fillna(999).astype(int)
counts["LABEL3"]       = counts["LABEL3"].fillna("Unknown")
counts["y_label"]      = counts["CLC_CODE"] + " - " + counts["LABEL3"]
counts["pct"]          = counts["count"] / counts["count"].sum() * 100
# Sort by CLC code; reverse for barh (lowest code at top)
counts = counts.sort_values("clc_code_int", ascending=False).reset_index(drop=True)

# ---------------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 16,
    "axes.labelsize": 15,
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
})

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(
    counts["y_label"],
    counts["count"],
    color=counts["COLOR_HEX"],
    edgecolor="#333333",
    linewidth=0.6,
    zorder=3,
)

ax.tick_params(axis="y", labelsize=13)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.set_xlabel("Number of NHDAs", fontsize=12)
# ax.set_title("Distribution of NHDAs by CORINE Land Cover Level 3 (2012)", fontsize=12, pad=12)

ax.grid(axis="x", color="#cccccc", linewidth=0.6, zorder=0)
ax.set_axisbelow(True)

for side in ["top", "right"]:
    ax.spines[side].set_visible(False)

x_offset = counts["count"].max() * 0.012
for bar, val in zip(bars, counts["count"]):
    ax.text(
        bar.get_width() + x_offset,
        bar.get_y() + bar.get_height() / 2,
        str(val),
        ha="left", va="center", fontsize=11, color="#333333",
    )

fig.tight_layout()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(OUTPUT_PLOT, bbox_inches="tight", facecolor="white", format="jpg")
plt.show()

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
total = counts["count"].sum()
print(f"Total NHDAs: {total}\n")

# Count + share per CLC class
summary = counts.sort_values("clc_code_int")[["y_label", "count", "pct"]].copy()
summary["pct"] = summary["pct"].map(lambda x: f"{x:.1f}%")
print(summary.to_string(index=False))

# nhda_sur_clc_2021_pct statistics per CLC class
pct_stats = (
    nhda.groupby("clc_code")["nhda_sur_clc_2021_pct"]
    .agg(
        nhda_count="size",
        mean="mean",
        std="std",
        min="min",
        p5=lambda x: x.quantile(0.05),
        p95=lambda x: x.quantile(0.95),
        max="max",
    )
    .reset_index()
    .rename(columns={"clc_code": "CLC_CODE"})
    .merge(legend_df[["CLC_CODE", "clc_code_int", "LABEL3"]], on="CLC_CODE", how="left")
    .sort_values("clc_code_int")
)
pct_stats["y_label"] = pct_stats["CLC_CODE"] + " - " + pct_stats["LABEL3"]
pct_stats = pct_stats[["y_label", "nhda_count", "mean", "std", "min", "p5", "p95", "max"]]
for col in ["mean", "std", "min", "p5", "p95", "max"]:
    pct_stats[col] = pct_stats[col].map(lambda x: f"{x:.1f}")

print("\nnhda_sur_clc_2021_pct per CLC class:")
print(pct_stats.to_string(index=False))
print(f"\nSaved: {OUTPUT_PLOT}")
